# ChemBERTa–SLICES: adaptación conservadora del tokenizer

## Propósito

Este notebook corresponde a la etapa 2B. Parte del tokenizer ChemBERTa Zinc auditado en `01_chemberta_tokenizer_audit.ipynb`, identifica lexemas SLICES fragmentados, compara dos opciones de `AddedToken`, adapta una copia conservando todos los IDs originales, repite el audit sobre el corpus completo y produce evidencia BASE vs ADAPTED.

No carga pesos Transformer, no entrena MLM/DAPT, no hace fine-tuning, no modifica embeddings y no crea un dataset de polimorfos. La adaptación modifica exclusivamente el **tokenizer**.

## Cómo interpretar este experimento

- Las cadenas SLICES se comparan tal como aparecen, con normalización de whitespace solamente.
- El baseline y el tokenizer final se miden sobre todas las filas utilizables, sin padding y sin truncamiento.
- Cada lexema se evalúa tanto aislado como con espacio precedente, el contexto habitual en cadenas SLICES. Al construir el tokenizer, también se mide si `lstrip=True` absorbe el whitespace y qué pasa con los lexemas al inicio de una cadena.
- Se compara `AddedToken(lstrip=False)` con `AddedToken(lstrip=True)`. `lstrip=True` puede absorber el whitespace precedente; el experimento verifica el resultado en vez de asumirlo.
- Una menor longitud es evidencia de eficiencia de tokenización, no una prueba automática de mejor representación química o mejor modelo.

El audit previo observó aproximadamente 99 998 cadenas, 320 lexemas, vocabulario base 767, fertilidad global 2.758 y 22.35 % de secuencias sobre 512 tokens. Esos valores son referencias: este notebook vuelve a medir el dataset y presenta las diferencias observadas.


## 1. Rutas del proyecto y carpeta de ejecución

Se busca hacia arriba hasta localizar simultáneamente `data/` y `audit/`. Todas las ejecuciones reutilizan `outputs/adaptation_run_latest/`. Los CSV, PNG y JSON de esta etapa se sobrescriben en esa carpeta con los resultados actuales; los outputs del audit 01 están en otra ubicación y se preservan. El tokenizer aprobado, si corresponde, se guarda aparte bajo `audit/tokenizers/`; aquí nunca se guarda un modelo Transformer.

In [ ]:
from pathlib import Path
from datetime import datetime
import time

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data").is_dir() and (candidate / "audit").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing both data/ and audit/")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
AUDIT_DIR = PROJECT_ROOT / "audit"
NOTEBOOK_DIR = AUDIT_DIR / "notebooks"
OUTPUTS_ROOT = NOTEBOOK_DIR / "outputs"
TOKENIZER_ROOT = AUDIT_DIR / "tokenizers"
TOKENIZER_DIR = TOKENIZER_ROOT / "chemberta_zinc_slices_adapted"
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
TOKENIZER_ROOT.mkdir(parents=True, exist_ok=True)

# Carpeta persistente: cada ejecución vuelve a generar los mismos nombres de archivo
# y sobrescribe las tablas, figuras y resúmenes producidos por este notebook.
OUTPUT_DIR = OUTPUTS_ROOT / "adaptation_run_latest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("AUDIT_DIR:", AUDIT_DIR)
print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("OUTPUT_DIR (reutilizada; los outputs de este notebook se sobrescriben):", OUTPUT_DIR)
print("TOKENIZER_DIR (solo si se acepta):", TOKENIZER_DIR)


## 2. Configuración y reproducibilidad

Se carga el mismo checkpoint base del audit 01. `MODEL_REVISION` se puede fijar a un commit de Hugging Face para reproducibilidad estricta; si se deja en `None`, el JSON registrará lo que exponga el tokenizer cargado. `BATCH_SIZE` regula memoria/velocidad, y la muestra A/B es reproducible con `RANDOM_SEED`.

In [ ]:
import ast
import json
import hashlib
import math
import re
import sys
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

import transformers
from transformers import AutoTokenizer, AddedToken
from transformers.utils import logging as hf_logging

BASE_MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"
MODEL_REVISION = None  # Para fijar un snapshot, asigna aquí un commit SHA de Hugging Face.
RANDOM_SEED = 42
BATCH_SIZE = 512
STRATEGY_SAMPLE_SIZE = 6000
RELOAD_TEST_SIZE = 1000
PLOT_SAMPLE_SIZE = 20000
CONTEXT_LIMIT = 512

ELEMENT_RE = re.compile(r"^[A-Z][a-z]?$")
INTEGER_RE = re.compile(r"^\d+$")
PERIODIC_RE = re.compile(r"^[o+\-]{3}$")
np.random.seed(RANDOM_SEED)

print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Base model:", BASE_MODEL_NAME)
print("Requested revision:", MODEL_REVISION)
print("Random seed:", RANDOM_SEED)


## 3. Cargar el mismo corpus SLICES que el audit 01

Se usa `alexandria_clean_all.csv`, no la tabla de candidatos a polimorfos. Si el audit 01 dejó su ruta de dataset en `tokenizer_audit_summary.json`, se reutiliza esa ruta cuando siga disponible. En otro caso, una copia única bajo `data/` se acepta; si hay varias, el notebook se detiene y exige seleccionar explícitamente una. Solo se carga `slices` y metadata estructural; no se leen targets energéticos.

In [ ]:
# Prefer the latest audit 01 summary inside a run folder; the root-level legacy
# duplicate is accepted only as a fallback for older project layouts.
audit01_summary_candidates = sorted(
    OUTPUTS_ROOT.glob("run_*/tokenizer_audit_summary.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
legacy_summary = OUTPUTS_ROOT / "tokenizer_audit_summary.json"
if legacy_summary.is_file():
    audit01_summary_candidates.append(legacy_summary)
BASE_AUDIT_SUMMARY_PATH = audit01_summary_candidates[0] if audit01_summary_candidates else None
base_audit_summary_previous = None
if BASE_AUDIT_SUMMARY_PATH is not None:
    with BASE_AUDIT_SUMMARY_PATH.open(encoding="utf-8") as handle:
        base_audit_summary_previous = json.load(handle)
    print("Audit 01 baseline summary:", BASE_AUDIT_SUMMARY_PATH)
else:
    print("No se encontró el summary del audit 01; se recalculará el baseline desde el CSV.")

preferred_dataset = DATA_DIR / "notebooks_processing" / "outputs" / "alexandria_clean_all.csv"
if base_audit_summary_previous:
    previous_dataset = base_audit_summary_previous.get("dataset")
    if previous_dataset and Path(previous_dataset).is_file():
        INPUT_CSV = Path(previous_dataset).resolve()
    elif preferred_dataset.is_file():
        INPUT_CSV = preferred_dataset.resolve()
    else:
        INPUT_CSV = None
elif preferred_dataset.is_file():
    INPUT_CSV = preferred_dataset.resolve()
else:
    INPUT_CSV = None

if INPUT_CSV is None:
    dataset_matches = sorted({p.resolve() for p in DATA_DIR.rglob("alexandria_clean_all.csv")})
    if not dataset_matches:
        raise FileNotFoundError("No se encontró alexandria_clean_all.csv debajo de data/.")
    if len(dataset_matches) == 1:
        INPUT_CSV = dataset_matches[0]
    else:
        # Si existen copias en carpetas de figuras/ejecuciones, comprobar que sean
        # byte a byte idénticas. En ese caso preferir la ruta del output de limpieza.
        # Si difieren, detenerse para que el usuario elija el corpus correcto.
        def sha256_file(path, chunk_size=8 * 1024 * 1024):
            digest = hashlib.sha256()
            with path.open("rb") as handle:
                for chunk in iter(lambda: handle.read(chunk_size), b""):
                    digest.update(chunk)
            return digest.hexdigest()
        dataset_hashes = {path: sha256_file(path) for path in dataset_matches}
        if len(set(dataset_hashes.values())) != 1:
            details = "\n".join(f"- {path} (sha256={digest})" for path, digest in dataset_hashes.items())
            raise RuntimeError("Hay copias distintas de alexandria_clean_all.csv; define INPUT_CSV manualmente.\n" + details)
        cleaning_outputs = [p for p in dataset_matches if "cleaning_run_" in str(p)]
        INPUT_CSV = (cleaning_outputs or dataset_matches)[0]
        print(f"Se encontraron {len(dataset_matches)} copias idénticas (SHA-256); se usa la salida canónica de limpieza: {INPUT_CSV}")

SOURCE_COLUMNS = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()
REQUIRED_COLUMNS = ["mat_id", "slices", "composition_key", "nsites"]
missing_required = [c for c in REQUIRED_COLUMNS if c not in SOURCE_COLUMNS]
if missing_required:
    raise ValueError(f"Faltan columnas requeridas en el corpus: {missing_required}")
OPTIONAL_COLUMNS = [c for c in ["reduced_formula", "n_elements", "spg"] if c in SOURCE_COLUMNS]
dataset_columns = REQUIRED_COLUMNS + OPTIONAL_COLUMNS
df_source = pd.read_csv(INPUT_CSV, usecols=dataset_columns, low_memory=False)

def normalize_slices(text):
    return " ".join(str(text).split())

usable = df_source["slices"].notna() & df_source["slices"].astype("string").str.strip().ne("")
df = df_source.loc[usable].copy().reset_index(drop=False).rename(columns={"index": "source_row_index"})
df["slices_normalized"] = df["slices"].map(normalize_slices)
df = df.loc[df["slices_normalized"].ne("")].reset_index(drop=True)
df["slices_whitespace_tokens"] = df["slices_normalized"].map(lambda x: len(x.split())).astype("int32")
df["slices_char_length"] = df["slices_normalized"].str.len().astype("int32")

print("Dataset:", INPUT_CSV)
print("Source rows:", len(df_source), "| usable SLICES rows:", len(df))
print("Columns loaded (energy targets excluded):", df.columns.tolist())
print("Duplicate mat_id rows:", int(df["mat_id"].duplicated(keep=False).sum()))
print("Duplicate SLICES rows:", int(df["slices_normalized"].duplicated(keep=False).sum()))
display(df[["mat_id", "composition_key", "slices_whitespace_tokens"]].head())


## 4. Cargar tokenizer BASE y congelar sus invariantes

Se carga únicamente `AutoTokenizer`. Antes de cambiar nada se guardan `get_vocab()` (token → ID), longitud efectiva, tamaño base, IDs de tokens especiales, nombre de clase, límite de contexto y mapa de special tokens. Este snapshot es la referencia para demostrar que cada token original conserva su ID.

In [ ]:
load_kwargs = {"revision": MODEL_REVISION} if MODEL_REVISION is not None else {}
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, **load_kwargs)
base_vocab = base_tokenizer.get_vocab().copy()
base_len = len(base_tokenizer)
base_vocab_size = int(base_tokenizer.vocab_size)
SPECIAL_ID_NAMES = ["bos_token_id", "eos_token_id", "pad_token_id", "unk_token_id", "mask_token_id", "cls_token_id", "sep_token_id"]
base_special_ids = {name: getattr(base_tokenizer, name, None) for name in SPECIAL_ID_NAMES}
base_special_tokens_map = dict(base_tokenizer.special_tokens_map)
base_model_max_length = getattr(base_tokenizer, "model_max_length", None)

print("Class:", base_tokenizer.__class__.__name__)
print("vocab_size (base model vocab):", base_vocab_size)
print("len(tokenizer) (vocab + added tokens):", base_len)
print("model_max_length:", base_model_max_length)
print("special_tokens_map:")
print(base_special_tokens_map)
print("special token IDs:")
display(pd.DataFrame([{"name": k, "id": v} for k,v in base_special_ids.items()]))


## 5. Recalcular el baseline completo y contrastarlo con el audit 01

El baseline se recalcula sobre todas las secuencias con `truncation=False`, `padding=False`; no se copian cifras de referencia al resultado. También se comparan las métricas principales con el JSON del audit 01 si está disponible. Una diferencia puede indicar un cambio de dataset, revisión del tokenizer o versiones; se reporta, no se oculta ni se convierte en un valor esperado artificial.

In [ ]:
def batch_token_ids(tokenizer, texts, add_special_tokens=False, batch_size=BATCH_SIZE, desc="Tokenizando"):
    lengths=np.empty(len(texts),dtype=np.int32)
    unk_counts=np.zeros(len(texts),dtype=np.int32)
    token_counter=Counter()
    unk_id=tokenizer.unk_token_id
    old_level=hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()  # Los avisos de >max_length son esperados en este audit.
    try:
        for start in tqdm(range(0,len(texts),batch_size),desc=desc,unit="batch"):
            stop=min(start+batch_size,len(texts))
            encoded=tokenizer(
                texts[start:stop], add_special_tokens=add_special_tokens,
                truncation=False, padding=False,
                return_attention_mask=False, return_token_type_ids=False,
            )
            rows=encoded["input_ids"]
            lengths[start:stop]=[len(ids) for ids in rows]
            token_counter.update(token_id for ids in rows for token_id in ids)
            if unk_id is not None:
                unk_counts[start:stop]=[sum(token_id==unk_id for token_id in ids) for ids in rows]
    finally:
        hf_logging.set_verbosity(old_level)
    return lengths,unk_counts,token_counter

texts=df["slices_normalized"].tolist()
base_content_lengths,base_unk_counts,base_content_id_counter=batch_token_ids(
    base_tokenizer,texts,add_special_tokens=False,desc="BASE content"
)
base_full_lengths,base_full_unk_counts,base_full_id_counter=batch_token_ids(
    base_tokenizer,texts,add_special_tokens=True,desc="BASE with specials"
)
base_unk_id=base_tokenizer.unk_token_id
base_metrics={
    "vocab_size":base_len,
    "median_tokens":float(np.median(base_full_lengths)),
    "p95_tokens":float(np.quantile(base_full_lengths,.95)),
    "p99_tokens":float(np.quantile(base_full_lengths,.99)),
    "max_tokens":int(base_full_lengths.max()),
    "global_fertility":float(base_content_lengths.sum()/df["slices_whitespace_tokens"].sum()),
    "mean_fertility":float(np.mean(base_content_lengths/df["slices_whitespace_tokens"].to_numpy())),
    "median_fertility":float(np.median(base_content_lengths/df["slices_whitespace_tokens"].to_numpy())),
    "p95_fertility":float(np.quantile(base_content_lengths/df["slices_whitespace_tokens"].to_numpy(),.95)),
    "percent_over_128":float(100*np.mean(base_full_lengths>128)),
    "percent_over_256":float(100*np.mean(base_full_lengths>256)),
    "percent_over_512":float(100*np.mean(base_full_lengths>512)),
    "percent_over_1024":float(100*np.mean(base_full_lengths>1024)),
    "percent_over_2048":float(100*np.mean(base_full_lengths>2048)),
    "unk_rate":float(100*np.mean(base_unk_counts>0)) if base_unk_id is not None else None,
}
print("BASE metrics recalculated:")
display(pd.Series(base_metrics,name="observed"))
if base_audit_summary_previous:
    previous=base_audit_summary_previous
    compare_rows=[]
    for current_key,previous_key in [
        ("median_tokens","median_hf_tokens"),("p95_tokens","p95_hf_tokens"),
        ("p99_tokens","p99_hf_tokens"),("global_fertility","global_fertility"),
        ("percent_over_512","percent_over_512"),("percent_over_1024","percent_over_1024"),
    ]:
        if previous.get(previous_key) is not None:
            current=base_metrics[current_key]
            old=float(previous[previous_key])
            compare_rows.append({"metric":current_key,"audit_01":old,"recalculated":current,
                                 "absolute_difference":current-old})
    baseline_reconciliation=pd.DataFrame(compare_rows)
    display(baseline_reconciliation)
else:
    baseline_reconciliation=pd.DataFrame(columns=["metric","audit_01","recalculated","absolute_difference"])
    print("No se encontró el JSON de audit 01; el baseline se midió desde el corpus actual.")


## 6. Reconstruir el vocabulario lexical y contextos observados

Se cuenta cada lexema whitespace sobre el corpus completo. Además se separa cuántas veces aparece como primer lexema y cuántas veces aparece después de un espacio. Esa separación permite ponderar la medida contextual con la posición real de las ocurrencias, en lugar de tratar todos los tokens únicos por igual. Se conserva la categoría `other`.

In [ ]:
def classify_slices_token(token):
    if ELEMENT_RE.fullmatch(token): return "element"
    if INTEGER_RE.fullmatch(token): return "integer"
    if PERIODIC_RE.fullmatch(token): return "periodic_direction"
    return "other"

lexical_counter=Counter()
first_position_counter=Counter()
later_position_counter=Counter()
for text in df["slices_normalized"]:
    tokens=text.split()
    lexical_counter.update(tokens)
    if tokens:
        first_position_counter.update([tokens[0]])
        later_position_counter.update(tokens[1:])

total_lexical_occurrences=int(sum(lexical_counter.values()))
n_unique_slices_tokens=len(lexical_counter)
lexical_vocab=pd.DataFrame([
    {"token":token,"frequency":frequency,
     "percentage":100*frequency/total_lexical_occurrences,
     "token_type":classify_slices_token(token),
     "first_position_frequency":first_position_counter[token],
     "later_position_frequency":later_position_counter[token]}
    for token,frequency in lexical_counter.items()
]).sort_values("frequency",ascending=False).reset_index(drop=True)
print("Sequences:",len(df))
print("Unique lexical SLICES tokens:",n_unique_slices_tokens)
print("Lexical occurrences:",total_lexical_occurrences)
display(lexical_vocab.head(30))


## 7. Fragmentación BASE: token aislado y token con whitespace precedente

Para cada lexema se guardan IDs/tokens BASE en aislamiento (`token`) y en posición whitespace-prefijada (`" " + token`). Para este tokenizer RoBERTa ByteLevel, la segunda prueba caracteriza el caso normal de cada lexema dentro de SLICES. También se tokenizan frases cortas de inicio, medio y final para dejar ejemplos inspeccionables. La longitud contextual se usa para decidir adaptación; no se infiere a partir de una sola cadena aislada.

In [ ]:
base_fragmentation_rows=[]
for token in tqdm(lexical_vocab["token"],desc="BASE: lexemas únicos",unit="token"):
    isolated_ids=base_tokenizer.encode(token,add_special_tokens=False)
    prefixed_ids=base_tokenizer.encode(" "+token,add_special_tokens=False)
    middle_text=f"O {token} 0"
    final_text=f"0 1 {token}"
    middle_ids=base_tokenizer.encode(middle_text,add_special_tokens=False)
    final_ids=base_tokenizer.encode(final_text,add_special_tokens=False)
    base_fragmentation_rows.append({
        "token":token,
        "frequency":int(lexical_counter[token]),
        "token_type":classify_slices_token(token),
        "base_n_subtokens":len(isolated_ids),
        "base_token_ids":json.dumps([int(x) for x in isolated_ids]),
        "base_tokens":json.dumps(base_tokenizer.convert_ids_to_tokens(isolated_ids),ensure_ascii=False),
        "base_context_subtokens":len(prefixed_ids),
        "base_context_token_ids":json.dumps([int(x) for x in prefixed_ids]),
        "base_context_tokens":json.dumps(base_tokenizer.convert_ids_to_tokens(prefixed_ids),ensure_ascii=False),
        "base_middle_example":json.dumps(base_tokenizer.convert_ids_to_tokens(middle_ids),ensure_ascii=False),
        "base_final_example":json.dumps(base_tokenizer.convert_ids_to_tokens(final_ids),ensure_ascii=False),
        "first_position_frequency":int(first_position_counter[token]),
        "later_position_frequency":int(later_position_counter[token]),
    })
base_fragmentation=pd.DataFrame(base_fragmentation_rows)
lexical_vocab=lexical_vocab.merge(base_fragmentation,on=["token","frequency","token_type","first_position_frequency","later_position_frequency"],how="left",validate="one_to_one")
display(lexical_vocab[["token","token_type","frequency","base_n_subtokens","base_tokens","base_context_subtokens","base_context_tokens"]].head(30))


## 8. Contextual alignment BASE y whitespace marker `Ġ`

Se reportan dos medidas distintas: cobertura aislada (lexema sin espacio, comparable con el 31.72 % de elementos y 83.16 % de enteros del audit previo) y alineación contextual. La tasa contextual cuenta qué parte de las ocurrencias corresponde a una sola pieza en su contexto: primer lexema medido sin prefijo; los demás medidos con espacio precedente. El cálculo se pondera por ocurrencias reales. Para el indicador `Ġ` contamos el ID literal de ese token especial de whitespace, tanto sobre contenido como en el stream completo. También distinguimos tokens BPE que empiezan por `Ġ`; ambos diagnósticos ayudan a explicar el overhead.

In [ ]:
def weighted_lexical_metrics(fragmentation, stage_prefix):
    total=0
    aligned=0
    type_acc=Counter()
    type_aligned=Counter()
    weighted_subtokens=Counter()
    for row in fragmentation.to_dict(orient="records"):
        token=row["token"]; kind=row["token_type"]
        first=int(row["first_position_frequency"]); later=int(row["later_position_frequency"])
        first_len=int(row[f"{stage_prefix}_n_subtokens"])
        later_len=int(row[f"{stage_prefix}_context_subtokens"])
        n=first+later
        total+=n; type_acc[kind]+=n
        aligned_count=(first if first_len==1 else 0)+(later if later_len==1 else 0)
        aligned+=aligned_count; type_aligned[kind]+=aligned_count
        weighted_subtokens[kind]+=first*first_len+later*later_len
        weighted_subtokens["global"]+=first*first_len+later*later_len
        type_acc["global"]+=n; type_aligned["global"]+=aligned_count
    output={kind:(100*type_aligned[kind]/type_acc[kind] if type_acc[kind] else np.nan)
            for kind in ["global","element","integer","periodic_direction","other"]}
    mean_subtokens={kind:(weighted_subtokens[kind]/type_acc[kind] if type_acc[kind] else np.nan)
                    for kind in ["global","element","integer","periodic_direction","other"]}
    return output,mean_subtokens

base_alignment_by_type,base_context_mean_by_type=weighted_lexical_metrics(base_fragmentation,"base")

def isolated_alignment_metrics(fragmentation, length_column):
    """Frequency-weighted single-piece rate when a lexeme is encoded by itself."""
    result={}
    for kind in ["global","element","integer","periodic_direction","other"]:
        group=fragmentation if kind=="global" else fragmentation.loc[fragmentation["token_type"]==kind]
        denominator=group["frequency"].sum()
        numerator=group.loc[group[length_column].eq(1),"frequency"].sum()
        result[kind]=100*numerator/denominator if denominator else np.nan
    return result

base_isolated_alignment_by_type=isolated_alignment_metrics(base_fragmentation,"base_n_subtokens")
base_space_marker_id=base_tokenizer.convert_tokens_to_ids("Ġ")
base_content_space_marker_frequency=(base_content_id_counter[base_space_marker_id]
                                     if base_space_marker_id is not None else 0)
base_content_id_total=int(sum(base_content_id_counter.values()))
base_space_marker_fraction=(base_content_space_marker_frequency/base_content_id_total
                            if base_content_id_total else 0.0)
base_token_strings=Counter({base_tokenizer.convert_ids_to_tokens(int(i)):count
                            for i,count in base_content_id_counter.items()})
base_prefixed_g_frequency=sum(count for token,count in base_token_strings.items() if token.startswith("Ġ"))
base_whitespace_marker_report=pd.DataFrame([
    {"measure":"standalone_Ġ_id_occurrences","frequency":int(base_content_space_marker_frequency),"fraction_of_content_ids":base_space_marker_fraction},
    {"measure":"BPE_piece_starts_with_Ġ_occurrences","frequency":int(base_prefixed_g_frequency),"fraction_of_content_ids":base_prefixed_g_frequency/base_content_id_total if base_content_id_total else 0.0},
])
print("BASE lexical alignment by occurrence:")
display(pd.Series(base_alignment_by_type,name="single-piece alignment %"))
print("BASE Ġ diagnostics:")
display(base_whitespace_marker_report)


## 9. Seleccionar unidades candidatas sin redundancia evidente

Un lexema necesita adaptación si aparece fragmentado aislado o si el contexto con espacio precedente requiere más de una pieza. Las unidades que ya forman una sola pieza tanto al inicio como en contexto no se vuelven a agregar. Se conserva una razón explícita por candidato. Esta selección inicial se somete a las dos estrategias A/B; la estrategia ganadora se vuelve a validar sobre el corpus completo.

In [ ]:
lexical_vocab["needs_adaptation"]=(
    lexical_vocab["base_n_subtokens"].gt(1)
    | lexical_vocab["base_context_subtokens"].gt(1)
)
def adaptation_reason(row):
    reasons=[]
    if row["base_n_subtokens"]>1: reasons.append("aislado se divide en varias piezas")
    if row["base_context_subtokens"]>1: reasons.append("con whitespace precedente requiere varias piezas (overhead Ġ/contexto)")
    return "; ".join(reasons) if reasons else "ya está alineado como una pieza aislada y contextual"
lexical_vocab["reason_for_adaptation"]=lexical_vocab.apply(adaptation_reason,axis=1)
tokens_selected_for_adaptation=(
    lexical_vocab.loc[lexical_vocab["needs_adaptation"]]
    .sort_values(["frequency","base_context_subtokens"],ascending=[False,False])
    [["token","token_type","frequency","base_n_subtokens","base_tokens","reason_for_adaptation"]]
    .copy()
)
print("Unique SLICES tokens needing adaptation:",len(tokens_selected_for_adaptation),"/",n_unique_slices_tokens)
display(tokens_selected_for_adaptation.head(40))


## 10. Experimento A/B de `AddedToken` y muestreo corto/medio/largo

Se preparan dos tokenizers independientes desde el mismo checkpoint y se añaden exactamente los mismos lexemas candidatos. **A** usa `lstrip=False`; **B** usa `lstrip=True`. Ambas pruebas mantienen `rstrip=False`, `normalized=False` y `single_word=True`; este último evita que un token léxico corto se case dentro de otro lexema más largo. `single_word` se mantiene constante para aislar el efecto del whitespace.

La comparación usa: todos los lexemas únicos para alineación, más 6 000 secuencias elegidas reproduciblemente entre terciles cortos, medios y largos. La muestra no reemplaza el audit completo; solo determina la estrategia. Se comprueban invariantes del vocabulario original en cada copia antes de permitir que compita.

In [ ]:
length_rank=df["slices_whitespace_tokens"].rank(method="first",pct=True)
df["length_band_for_strategy_sample"]=pd.cut(
    length_rank,bins=[0,1/3,2/3,1.000001],labels=["short","medium","long"],include_lowest=True
)
parts=[]
per_band=max(1,STRATEGY_SAMPLE_SIZE//3)
for band in ["short","medium","long"]:
    group=df.loc[df["length_band_for_strategy_sample"]==band]
    parts.append(group.sample(n=min(per_band,len(group)),random_state=RANDOM_SEED))
strategy_sample=pd.concat(parts).drop_duplicates(subset="source_row_index").sort_values("source_row_index").reset_index(drop=True)
strategy_texts=strategy_sample["slices_normalized"].tolist()
print("Strategy sample rows:",len(strategy_sample))
display(strategy_sample.groupby("length_band_for_strategy_sample",observed=True)["slices_whitespace_tokens"].describe())

candidate_tokens=tokens_selected_for_adaptation["token"].tolist()
def make_trial_tokenizer(lstrip_value):
    trial=AutoTokenizer.from_pretrained(BASE_MODEL_NAME,**load_kwargs)
    added=[AddedToken(token,lstrip=lstrip_value,rstrip=False,normalized=False,single_word=True)
           for token in candidate_tokens]
    n_returned=trial.add_tokens(added,special_tokens=False)
    trial_vocab=trial.get_vocab()
    changed=[token for token,token_id in base_vocab.items() if trial_vocab.get(token)!=token_id]
    specials={name:getattr(trial,name,None) for name in SPECIAL_ID_NAMES}
    special_changed=[name for name,value in base_special_ids.items() if specials.get(name)!=value]
    return trial,n_returned,changed,special_changed

strategy_trial_rows=[]
strategy_trial_fragmentations={}
for strategy_name,lstrip_value in [("A_lstrip_false",False),("B_lstrip_true",True)]:
    trial,n_returned,changed,special_changed=make_trial_tokenizer(lstrip_value)
    trial_fragment=base_fragmentation[[
        "token","frequency","token_type","first_position_frequency","later_position_frequency"
    ]].copy()
    isolated_lengths=[]; contextual_lengths=[]
    for token in trial_fragment["token"]:
        isolated_lengths.append(len(trial.encode(token,add_special_tokens=False)))
        contextual_lengths.append(len(trial.encode(" "+token,add_special_tokens=False)))
    trial_fragment[f"{strategy_name}_n_subtokens"]=isolated_lengths
    trial_fragment[f"{strategy_name}_context_subtokens"]=contextual_lengths
    alignment,mean_by_type=weighted_lexical_metrics(trial_fragment,strategy_name)
    sample_content, sample_unk,_=batch_token_ids(
        trial,strategy_texts,add_special_tokens=False,desc=f"Estrategia {strategy_name}"
    )
    sample_full, sample_full_unk,_=batch_token_ids(
        trial,strategy_texts,add_special_tokens=True,desc=f"Longitudes {strategy_name}"
    )
    sample_lexical=strategy_sample["slices_whitespace_tokens"].to_numpy(dtype=float)
    sample_fertility=sample_content/sample_lexical
    unk_rate=float(100*np.mean(sample_unk>0)) if trial.unk_token_id is not None else 0.0
    invariant_pass=(not changed and not special_changed and unk_rate <= base_metrics["unk_rate"])
    strategy_trial_rows.append({
        "strategy":strategy_name,"lstrip":lstrip_value,"added_tokens_returned":int(n_returned),
        "effective_vocab_len":len(trial),"original_ids_changed":len(changed),
        "special_ids_changed":len(special_changed),"sample_unk_rate":unk_rate,
        "sample_global_fertility":float(sample_content.sum()/sample_lexical.sum()),
        "sample_mean_fertility":float(sample_fertility.mean()),
        "sample_median_fertility":float(np.median(sample_fertility)),
        "sample_p95_fertility":float(np.quantile(sample_fertility,.95)),
        "sample_median_sequence_length":float(np.median(sample_full)),
        "sample_percent_over_512":float(100*np.mean(sample_full>512)),
        "global_lexical_alignment_percent":alignment["global"],
        "element_alignment_percent":alignment["element"],
        "integer_alignment_percent":alignment["integer"],
        "periodic_alignment_percent":alignment["periodic_direction"],
        "critical_trial_invariants_pass":invariant_pass,
    })
    strategy_trial_fragmentations[strategy_name]=trial_fragment
    del trial

strategy_comparison=pd.DataFrame(strategy_trial_rows)
display(strategy_comparison)


## 11. Elegir automáticamente la estrategia experimental

Solo compiten estrategias que conservan todos los IDs y special IDs originales y no empeoran UNK en la muestra. Entre ellas se usa un orden lexicográfico: (1) mayor alineación lexical ponderada, (2) menor fertilidad global, (3) menos secuencias sobre 512 y (4) menor tamaño del vocabulario efectivo. Así la decisión favorece primero que cada unidad SLICES sea una unidad contextual y luego considera longitud/contexto. El resultado y los valores A/B se guardan para inspección.

In [ ]:
eligible_trials=strategy_comparison.loc[strategy_comparison["critical_trial_invariants_pass"]].copy()
if eligible_trials.empty:
    raise RuntimeError("Ninguna estrategia A/B preserva invariantes; no se construirá tokenizer final.")
selected_trial=eligible_trials.sort_values(
    ["global_lexical_alignment_percent","sample_global_fertility","sample_percent_over_512","effective_vocab_len"],
    ascending=[False,True,True,True],kind="mergesort"
).iloc[0]
adaptation_strategy=str(selected_trial["strategy"])
selected_lstrip=bool(selected_trial["lstrip"])
print("Estrategia seleccionada:",adaptation_strategy)
print("lstrip:",selected_lstrip)
print("Criterio: alineación contextual ponderada; desempate por fertilidad, overflow >512 y vocabulario efectivo.")
print("Trial metrics:")
display(selected_trial.to_frame("selected_strategy"))


## 12. Construir la copia ADAPTED definitiva

El tokenizer final vuelve a cargar el checkpoint limpio y agrega las unidades seleccionadas con la opción ganadora. No se reconstruye BPE, no se tocan merges y no se agregan los tokens como special tokens. Tokens ya presentes en el vocabulario pueden conservar su ID original y recibir metadatos de `AddedToken`; solo unidades realmente nuevas aumentan la longitud efectiva del vocabulario.

In [ ]:
adapted_tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL_NAME,**load_kwargs)
added_token_objects=[
    AddedToken(token,lstrip=selected_lstrip,rstrip=False,normalized=False,single_word=True)
    for token in candidate_tokens
]
added_tokens_returned=adapted_tokenizer.add_tokens(added_token_objects,special_tokens=False)
adapted_vocab=adapted_tokenizer.get_vocab().copy()
adapted_len=len(adapted_tokenizer)
adapted_vocab_size=int(adapted_tokenizer.vocab_size)
new_lexical_tokens=[token for token in candidate_tokens if token not in base_vocab and token in adapted_vocab]
new_token_ids=[int(adapted_vocab[token]) for token in new_lexical_tokens]
n_added_tokens=adapted_len-base_len
print("Candidate SLICES lexemes:",len(candidate_tokens))
print("add_tokens return value:",added_tokens_returned)
print("Effective added vocabulary entries:",n_added_tokens)
print("Base len(tokenizer):",base_len,"| adapted len(tokenizer):",adapted_len)
print("Underlying adapted tokenizer.vocab_size:",adapted_vocab_size,"(base BPE vocab; added tokens excluded by this property)")
print("New lexeme ID min/max:",min(new_token_ids) if new_token_ids else None,max(new_token_ids) if new_token_ids else None)


## 13. Invariantes: proteger vocabulario y special IDs originales

Se compara el mapping entero `token → ID`, no solo el tamaño. Cualquier token original ausente o con ID diferente es fallo crítico. Se revisan por separado BOS/EOS/PAD/UNK/MASK/CLS/SEP; algunos nombres son aliases en RoBERTa, pero todos deben retener exactamente su valor. IDs realmente nuevos deben empezar en `base_len` o después.

In [ ]:
changed_original_tokens=[
    {"token":token,"base_id":int(token_id),"adapted_id":adapted_vocab.get(token)}
    for token,token_id in base_vocab.items() if adapted_vocab.get(token)!=token_id
]
n_original_ids_changed=len(changed_original_tokens)
adapted_special_ids={name:getattr(adapted_tokenizer,name,None) for name in SPECIAL_ID_NAMES}
changed_special_ids=[
    {"name":name,"base_id":base_special_ids[name],"adapted_id":adapted_special_ids.get(name)}
    for name in SPECIAL_ID_NAMES if adapted_special_ids.get(name)!=base_special_ids[name]
]
new_token_id_min=min(new_token_ids) if new_token_ids else None
new_token_id_max=max(new_token_ids) if new_token_ids else None
new_ids_after_base=(new_token_id_min is None or new_token_id_min>=base_len)
original_vocab_preserved=n_original_ids_changed==0
special_ids_preserved=len(changed_special_ids)==0
print("Original tokens whose IDs changed:",n_original_ids_changed)
print("Special token IDs changed:",len(changed_special_ids))
print("All new IDs >= original len(tokenizer):",new_ids_after_base)
assert original_vocab_preserved, "ADAPTATION FAILED: an original token ID changed or disappeared."
assert special_ids_preserved, "ADAPTATION FAILED: a special token ID changed."
assert new_ids_after_base, "ADAPTATION FAILED: a new lexical token received an ID below the base tokenizer length."


## 14. Guardar candidato, recargar y comprobar igualdad

Primero se guarda el candidato dentro de esta carpeta de ejecución, nunca encima de un tokenizer previamente aprobado. Después se recarga con `AutoTokenizer` y se comparan los IDs de 1 000 secuencias elegidas de forma reproducible. También se vuelven a comprobar tamaño efectivo, mapping original y special IDs. Solo si estas pruebas pasan puede considerarse guardar la ruta oficial.

In [ ]:
CANDIDATE_TOKENIZER_DIR=OUTPUT_DIR/"candidate_tokenizer"
adapted_tokenizer.save_pretrained(CANDIDATE_TOKENIZER_DIR)
reloaded=AutoTokenizer.from_pretrained(CANDIDATE_TOKENIZER_DIR)
reload_n=min(RELOAD_TEST_SIZE,len(df))
reload_sample=df.sample(n=reload_n,random_state=RANDOM_SEED)["slices_normalized"].tolist()
old_level=hf_logging.get_verbosity(); hf_logging.set_verbosity_error()
try:
    adapted_reload_ids=reloaded(reload_sample,add_special_tokens=False,truncation=False,padding=False,
                               return_attention_mask=False,return_token_type_ids=False)["input_ids"]
    adapted_direct_ids=adapted_tokenizer(reload_sample,add_special_tokens=False,truncation=False,padding=False,
                                         return_attention_mask=False,return_token_type_ids=False)["input_ids"]
finally:
    hf_logging.set_verbosity(old_level)
reload_equal=[left==right for left,right in zip(adapted_direct_ids,adapted_reload_ids)]
reload_reproducible=(len(reload_equal)==reload_n and all(reload_equal))
reloaded_vocab=reloaded.get_vocab()
reloaded_original_ids_preserved=all(reloaded_vocab.get(token)==token_id for token,token_id in base_vocab.items())
reloaded_special_ids={name:getattr(reloaded,name,None) for name in SPECIAL_ID_NAMES}
reloaded_special_ids_preserved=all(reloaded_special_ids[name]==base_special_ids[name] for name in SPECIAL_ID_NAMES)
print("Save/reload sample:",reload_n)
print("Encoding equality:",sum(reload_equal),"/",reload_n)
print("Reloaded len(tokenizer):",len(reloaded),"| expected:",adapted_len)
print("Reloaded original vocab IDs preserved:",reloaded_original_ids_preserved)
print("Reloaded special IDs preserved:",reloaded_special_ids_preserved)
assert reload_reproducible, "ADAPTATION FAILED: save/reload encoding mismatch."
assert len(reloaded)==adapted_len, "ADAPTATION FAILED: tokenizer size changed after reload."
assert reloaded_original_ids_preserved and reloaded_special_ids_preserved, "ADAPTATION FAILED after reload: ID invariants." 


## 15. FULL AUDIT ADAPTED sobre todas las secuencias

Se vuelve a tokenizar el corpus completo, tanto sin special tokens como con ellos. No se activa padding ni truncamiento. Se retienen por secuencia longitudes, UNK y fertilidad; los `input_ids` completos se descartan al terminar cada lote. Esto cuantifica el efecto real de adaptación sobre el contexto y no solo el rendimiento de lexemas seleccionados.

In [ ]:
adapted_content_lengths,adapted_unk_counts,adapted_content_id_counter=batch_token_ids(
    adapted_tokenizer,texts,add_special_tokens=False,desc="ADAPTED content"
)
adapted_full_lengths,adapted_full_unk_counts,adapted_full_id_counter=batch_token_ids(
    adapted_tokenizer,texts,add_special_tokens=True,desc="ADAPTED with specials"
)
lexical_denominator=df["slices_whitespace_tokens"].to_numpy(dtype=float)
base_fertility_per_sequence=base_content_lengths/lexical_denominator
adapted_fertility_per_sequence=adapted_content_lengths/lexical_denominator
adapted_unk_id=adapted_tokenizer.unk_token_id
base_unk_rate=(float(100*np.mean(base_unk_counts>0)) if base_unk_id is not None else None)
adapted_unk_rate=(float(100*np.mean(adapted_unk_counts>0)) if adapted_unk_id is not None else None)

df["base_hf_tokens_no_special"]=base_content_lengths
df["base_hf_tokens_with_special"]=base_full_lengths
df["base_fertility"]=base_fertility_per_sequence
df["base_unk_count"]=base_unk_counts
df["adapted_hf_tokens_no_special"]=adapted_content_lengths
df["adapted_hf_tokens_with_special"]=adapted_full_lengths
df["adapted_fertility"]=adapted_fertility_per_sequence
df["adapted_unk_count"]=adapted_unk_counts
df["adapted_unk_fraction"]=np.divide(adapted_unk_counts,adapted_content_lengths,
    out=np.zeros(len(df),dtype=float),where=adapted_content_lengths>0)
for limit in [128,256,512,1024,2048]:
    df[f"base_over_{limit}"]=base_full_lengths>limit
    df[f"adapted_over_{limit}"]=adapted_full_lengths>limit
print("BASE sequence count:",len(base_full_lengths),"| ADAPTED sequence count:",len(adapted_full_lengths))
print("Rows lost:",len(base_full_lengths)-len(adapted_full_lengths))


## 16. Base vs adapted: longitudes, fertilidad, UNK y contexto

Esta es la comparación nuclear. Para tokens especiales la longitud es la que el modelo recibiría; la fertilidad usa solo IDs de contenido divididos por lexemas whitespace SLICES. La columna `improvement` se interpreta como tokens/porcentaje **reducidos** para magnitudes donde menor es mejor, y como puntos porcentuales **ganados** para coberturas donde mayor es mejor.

In [ ]:
def length_metrics(content_lengths,full_lengths,fertility,unk_counts,tokenizer):
    return {
        "vocab_size":len(tokenizer),
        "median_hf_tokens":float(np.median(full_lengths)),
        "p95_hf_tokens":float(np.quantile(full_lengths,.95)),
        "p99_hf_tokens":float(np.quantile(full_lengths,.99)),
        "max_hf_tokens":int(np.max(full_lengths)),
        "global_fertility":float(content_lengths.sum()/lexical_denominator.sum()),
        "mean_fertility":float(np.mean(fertility)),
        "median_fertility":float(np.median(fertility)),
        "p95_fertility":float(np.quantile(fertility,.95)),
        "percent_over_128":float(100*np.mean(full_lengths>128)),
        "percent_over_256":float(100*np.mean(full_lengths>256)),
        "percent_over_512":float(100*np.mean(full_lengths>512)),
        "percent_over_1024":float(100*np.mean(full_lengths>1024)),
        "percent_over_2048":float(100*np.mean(full_lengths>2048)),
        "unk_rate":float(100*np.mean(unk_counts>0)) if tokenizer.unk_token_id is not None else None,
    }
base_final_metrics=length_metrics(base_content_lengths,base_full_lengths,base_fertility_per_sequence,base_unk_counts,base_tokenizer)
adapted_metrics=length_metrics(adapted_content_lengths,adapted_full_lengths,adapted_fertility_per_sequence,adapted_unk_counts,adapted_tokenizer)
base_alignment_by_type,base_context_mean_by_type=weighted_lexical_metrics(base_fragmentation,"base")
adapted_fragmentation=base_fragmentation[[
    "token","frequency","token_type","first_position_frequency","later_position_frequency"
]].copy()
adapted_iso=[]; adapted_ctx=[]; adapted_ids_iso=[]; adapted_tokens_iso=[]; adapted_ids_ctx=[]; adapted_tokens_ctx=[]
for token in tqdm(adapted_fragmentation["token"],desc="ADAPTED: contexto lexical",unit="token"):
    ids_iso=adapted_tokenizer.encode(token,add_special_tokens=False)
    ids_ctx=adapted_tokenizer.encode(" "+token,add_special_tokens=False)
    adapted_iso.append(len(ids_iso)); adapted_ctx.append(len(ids_ctx))
    adapted_ids_iso.append(json.dumps([int(x) for x in ids_iso]))
    adapted_tokens_iso.append(json.dumps(adapted_tokenizer.convert_ids_to_tokens(ids_iso),ensure_ascii=False))
    adapted_ids_ctx.append(json.dumps([int(x) for x in ids_ctx]))
    adapted_tokens_ctx.append(json.dumps(adapted_tokenizer.convert_ids_to_tokens(ids_ctx),ensure_ascii=False))
adapted_fragmentation["adapted_n_subtokens"]=adapted_iso
adapted_fragmentation["adapted_token_ids"]=adapted_ids_iso
adapted_fragmentation["adapted_tokens"]=adapted_tokens_iso
adapted_fragmentation["adapted_context_subtokens"]=adapted_ctx
adapted_fragmentation["adapted_context_token_ids"]=adapted_ids_ctx
adapted_fragmentation["adapted_context_tokens"]=adapted_tokens_ctx
base_vs_adapted_fragmentation=base_fragmentation.merge(
    adapted_fragmentation[["token","adapted_n_subtokens","adapted_token_ids","adapted_tokens",
                           "adapted_context_subtokens","adapted_context_token_ids","adapted_context_tokens"]],
    on="token",how="left",validate="one_to_one")
adapted_alignment_by_type,adapted_context_mean_by_type=weighted_lexical_metrics(base_vs_adapted_fragmentation,"adapted")
adapted_isolated_alignment_by_type=isolated_alignment_metrics(base_vs_adapted_fragmentation,"adapted_n_subtokens")

metric_rows=[]
metric_better_lower={"vocab_size":False,"median_hf_tokens":True,"p95_hf_tokens":True,"p99_hf_tokens":True,
 "max_hf_tokens":True,"global_fertility":True,"mean_fertility":True,"median_fertility":True,
 "p95_fertility":True,"percent_over_128":True,"percent_over_256":True,"percent_over_512":True,
 "percent_over_1024":True,"percent_over_2048":True,"unk_rate":True}
for metric,base_value in base_final_metrics.items():
    adapted_value=adapted_metrics[metric]
    if base_value is None or adapted_value is None:
        improvement=np.nan
    elif metric=="vocab_size":
        improvement=adapted_value-base_value
    elif metric_better_lower.get(metric,False):
        improvement=base_value-adapted_value
    else:
        improvement=adapted_value-base_value
    metric_rows.append({"metric":metric,"BASE":base_value,"ADAPTED":adapted_value,"improvement":improvement})
for token_type in ["element","integer","periodic_direction","other"]:
    metric_rows.append({"metric":f"{token_type}_isolated_one_token_percent","BASE":base_isolated_alignment_by_type[token_type],
                        "ADAPTED":adapted_isolated_alignment_by_type[token_type],
                        "improvement":adapted_isolated_alignment_by_type[token_type]-base_isolated_alignment_by_type[token_type]})
    metric_rows.append({"metric":f"{token_type}_contextual_alignment_percent","BASE":base_alignment_by_type[token_type],
                        "ADAPTED":adapted_alignment_by_type[token_type],
                        "improvement":adapted_alignment_by_type[token_type]-base_alignment_by_type[token_type]})
metric_rows.append({"metric":"global_isolated_one_token_percent","BASE":base_isolated_alignment_by_type["global"],
                    "ADAPTED":adapted_isolated_alignment_by_type["global"],
                    "improvement":adapted_isolated_alignment_by_type["global"]-base_isolated_alignment_by_type["global"]})
metric_rows.append({"metric":"global_contextual_alignment_percent","BASE":base_alignment_by_type["global"],
                    "ADAPTED":adapted_alignment_by_type["global"],
                    "improvement":adapted_alignment_by_type["global"]-base_alignment_by_type["global"]})
base_vs_adapted_metrics=pd.DataFrame(metric_rows)
display(base_vs_adapted_metrics)


## 17. Alineación contextual y fragmentación por clase

La alineación lexical se pondera por ocurrencias y distingue primer token de tokens precedidos por espacio. Si el lexema produce exactamente una pieza en esa posición, la ocurrencia se considera alineada. La media de subtokens por clase también se pondera por frecuencia. `absolute_improvement` para media de subtokens es reducción BASE−ADAPTED; para alineación es ganancia de puntos porcentuales ADAPTED−BASE. La mejora relativa de la media se expresa como fracción de reducción frente al baseline.

In [ ]:
token_type_rows=[]
for token_type in ["element","integer","periodic_direction","other"]:
    base_mean=base_context_mean_by_type[token_type]
    adapted_mean=adapted_context_mean_by_type[token_type]
    base_single=base_isolated_alignment_by_type[token_type]
    adapted_single=adapted_isolated_alignment_by_type[token_type]
    token_type_rows.append({
        "token_type":token_type,
        "occurrences":int(lexical_vocab.loc[lexical_vocab["token_type"]==token_type,"frequency"].sum()),
        "base_mean_subtokens":base_mean,
        "adapted_mean_subtokens":adapted_mean,
        "base_percent_single_token":base_single,
        "adapted_percent_single_token":adapted_single,
        "absolute_improvement":base_mean-adapted_mean,
        "relative_improvement":(base_mean-adapted_mean)/base_mean if base_mean else np.nan,
        "single_token_alignment_gain_percentage_points":adapted_single-base_single,
        "base_contextual_alignment_percent":base_alignment_by_type[token_type],
        "adapted_contextual_alignment_percent":adapted_alignment_by_type[token_type],
        "context_alignment_gain_percentage_points":adapted_alignment_by_type[token_type]-base_alignment_by_type[token_type],
    })
base_vs_adapted_token_types=pd.DataFrame(token_type_rows)
display(base_vs_adapted_token_types)


## 18. Overhead whitespace `Ġ`

Se cuentan las apariciones del ID correspondiente al token literal `Ġ` en todos los IDs de contenido. `space_marker_fraction` divide por el total de IDs de contenido; `space_marker_reduction_percent` es la reducción relativa de ese overhead. Se añade una medida complementaria de BPE piezas cuyo texto visible comienza con `Ġ`, porque algunas variantes representan espacio unido al siguiente token en vez de un token marcador separado.

In [ ]:
def space_marker_metrics(tokenizer,content_counter):
    marker_id=tokenizer.convert_tokens_to_ids("Ġ")
    count=int(content_counter[marker_id]) if marker_id is not None else 0
    total=int(sum(content_counter.values()))
    pieces=Counter({tokenizer.convert_ids_to_tokens(int(token_id)):frequency
                    for token_id,frequency in content_counter.items()})
    prefixed=sum(freq for piece,freq in pieces.items() if piece.startswith("Ġ"))
    return {"standalone_marker_id":marker_id,"standalone_marker_frequency":count,
            "standalone_marker_fraction":count/total if total else 0.0,
            "pieces_starting_with_marker_frequency":int(prefixed),
            "pieces_starting_with_marker_fraction":prefixed/total if total else 0.0,
            "total_content_ids":total}
base_space_metrics=space_marker_metrics(base_tokenizer,base_content_id_counter)
adapted_space_metrics=space_marker_metrics(adapted_tokenizer,adapted_content_id_counter)
space_marker_reduction_percent=(
    100*(base_space_metrics["standalone_marker_frequency"]-adapted_space_metrics["standalone_marker_frequency"])
    /base_space_metrics["standalone_marker_frequency"]
    if base_space_metrics["standalone_marker_frequency"] else 0.0
)
space_marker_comparison=pd.DataFrame([
    {"metric":key,"BASE":base_space_metrics[key],"ADAPTED":adapted_space_metrics[key],
     "improvement":base_space_metrics[key]-adapted_space_metrics[key]}
    for key in ["standalone_marker_frequency","standalone_marker_fraction",
                "pieces_starting_with_marker_frequency","pieces_starting_with_marker_fraction"]
])
print("Standalone Ġ reduction (%):",space_marker_reduction_percent)
display(space_marker_comparison)


## 19. Estructuras grandes: comparar longitud y overflow por `nsites`

Se agrupa `nsites` con los intervalos especificados. Para cada grupo se calculan mediana/P95 y tasa >512 en BASE y ADAPTED, además de fertilidad media. Esto permite ver si el ahorro de tokens beneficia también a estructuras grandes; no prueba causalidad ni se usa para excluir estructuras.

In [ ]:
nsites_bins=[0,5,10,20,40,80,np.inf]
nsites_labels=["1-5","6-10","11-20","21-40","41-80","81+"]
df["nsites_group"]=pd.cut(pd.to_numeric(df["nsites"],errors="coerce"),bins=nsites_bins,
                           labels=nsites_labels,include_lowest=True)
size_rows=[]
for group_name,group in df.groupby("nsites_group",observed=False):
    row={"nsites_group":str(group_name),"n_sequences":len(group)}
    for prefix in ["base","adapted"]:
        lengths=group[f"{prefix}_hf_tokens_with_special"]
        row[f"{prefix}_median_tokens"]=float(lengths.median()) if len(group) else np.nan
        row[f"{prefix}_P95"]=float(lengths.quantile(.95)) if len(group) else np.nan
        row[f"{prefix}_percent_over_512"]=100*float((lengths>512).mean()) if len(group) else np.nan
        row[f"{prefix}_fertility"]=float(group[f"{prefix}_fertility"].mean()) if len(group) else np.nan
    size_rows.append(row)
base_vs_adapted_by_nsites=pd.DataFrame(size_rows)
display(base_vs_adapted_by_nsites)


## 20. Tokens con mayor ahorro y fragmentación restante

`estimated_total_subtokens_saved` multiplica la reducción contextual por ocurrencia por la frecuencia del lexema. Es una atribución por unidad lexical y puede no coincidir exactamente con tokens de secuencia en caso de interacciones tokenizer, por eso se contrasta con la longitud real completa. La tabla de remaining incluye cualquier token que siga produciendo más de una pieza aislada/contextual; no se ocultan fallos residuales.

In [ ]:
top_tokenization_improvements=base_vs_adapted_fragmentation.copy()
def decode_json_tokens(value):
    try: return " | ".join(json.loads(value))
    except Exception: return str(value)
top_tokenization_improvements["base_tokenization"]=top_tokenization_improvements["base_context_tokens"].map(decode_json_tokens)
top_tokenization_improvements["adapted_tokenization"]=top_tokenization_improvements["adapted_context_tokens"].map(decode_json_tokens)
top_tokenization_improvements["subtokens_saved_per_occurrence"]=(
    top_tokenization_improvements["base_context_subtokens"]-top_tokenization_improvements["adapted_context_subtokens"]
)
top_tokenization_improvements["estimated_total_subtokens_saved"]=(
    top_tokenization_improvements["subtokens_saved_per_occurrence"]*top_tokenization_improvements["frequency"]
)
top_tokenization_improvements=top_tokenization_improvements.sort_values(
    ["estimated_total_subtokens_saved","frequency"],ascending=False
)
remaining_fragmented_tokens=base_vs_adapted_fragmentation.loc[
    (base_vs_adapted_fragmentation["adapted_n_subtokens"]>1)
    | (base_vs_adapted_fragmentation["adapted_context_subtokens"]>1)
].copy()
remaining_fragmented_tokens["reason_remaining_fragmented"]="adapted isolated/context encoding still has >1 piece"
remaining_fragmented_tokens=remaining_fragmented_tokens.sort_values("frequency",ascending=False)
print("Top token savings:")
display(top_tokenization_improvements[["token","token_type","frequency","base_context_subtokens","adapted_context_subtokens","estimated_total_subtokens_saved"]].head(30))
print("Remaining fragmented lexemes:",len(remaining_fragmented_tokens))
display(remaining_fragmented_tokens[["token","token_type","frequency","adapted_n_subtokens","adapted_context_subtokens","adapted_context_tokens"]].head(50))


## 21. Secuencias más largas del baseline

Se seleccionan las 100 secuencias con más IDs BASE (incluyendo specials) y se comparan con su longitud ADAPTED y sus fertilidades. El porcentaje de reducción tiene como denominador la longitud BASE. Se mantiene el SLICES lexical count para verificar que la diferencia proviene de tokenización, no de alterar el string.

In [ ]:
longest_sequences_comparison=df.nlargest(100,"base_hf_tokens_with_special")[[
    column for column in ["mat_id","reduced_formula","nsites","slices_whitespace_tokens",
       "base_hf_tokens_with_special","adapted_hf_tokens_with_special","base_fertility","adapted_fertility"]
    if column in df.columns
]].copy()
longest_sequences_comparison["tokens_saved"]=(
    longest_sequences_comparison["base_hf_tokens_with_special"]-longest_sequences_comparison["adapted_hf_tokens_with_special"]
)
longest_sequences_comparison["percent_reduction"]=(
    100*longest_sequences_comparison["tokens_saved"]/longest_sequences_comparison["base_hf_tokens_with_special"]
)
display(longest_sequences_comparison.head(15))


## 22. Simulación de truncamiento a 512

Para ambos tokenizers se simula el número de secuencias afectadas y los tokens que quedarían fuera. `fraction_retained = min(length,512)/length`. Las métricas incluyen media/mediana de la fracción sobre el corpus. Ninguna cadena se trunca de verdad y no se crea un dataset truncado.

In [ ]:
truncation_rows=[]
for stage,lengths in [("BASE",base_full_lengths),("ADAPTED",adapted_full_lengths)]:
    lost=np.maximum(lengths-512,0)
    affected=lengths>512
    retained=np.divide(np.minimum(lengths,512),lengths,out=np.ones(len(lengths),dtype=float),where=lengths>0)
    truncation_rows.append({
        "tokenizer":stage,"n_affected":int(affected.sum()),"percent_affected":100*float(affected.mean()),
        "mean_tokens_lost":float(lost.mean()),"median_tokens_lost":float(np.median(lost)),
        "P95_tokens_lost":float(np.quantile(lost,.95)),
        "mean_fraction_retained":float(retained.mean()),"median_fraction_retained":float(np.median(retained)),
    })
base_vs_adapted_truncation_512=pd.DataFrame(truncation_rows)
base_over512=base_metrics["percent_over_512"]
adapted_over512=adapted_metrics["percent_over_512"]
over512_reduction_pp=base_over512-adapted_over512
over512_relative_reduction=over512_reduction_pp/base_over512 if base_over512 else np.nan
display(base_vs_adapted_truncation_512)
print(">512 absolute reduction (percentage points):",over512_reduction_pp)
print(">512 relative reduction (%):",100*over512_relative_reduction)


## 23. Validación por tipo lexical y mejoras residuales

Se informa la fragmentación ponderada para elemento, integer, periodic direction y other; también se compara la cobertura one-token. La cobertura de alignment se calcula en contexto de whitespace para ocurrencias posteriores al primer lexema. El objetivo experimental global es al menos 95 %, pero valores inferiores se documentan sin intentar corregir métricas artificialmente.

In [ ]:
print("Tipo lexical: BASE vs ADAPTED")
display(base_vs_adapted_token_types)
print("Alineación global BASE:",base_alignment_by_type["global"])
print("Alineación global ADAPTED:",adapted_alignment_by_type["global"])
print("Tokens todavía fragmentados:",len(remaining_fragmented_tokens))


## 24. Figuras comparativas

Cada PNG se genera con Matplotlib a 180 dpi y `bbox_inches="tight"`. Histogramas, ECDF, tasas de overflow, clases lexicales, tamaño de estructura y ahorro muestran diferentes aspectos de la adaptación. En las distribuciones de longitud el eje visual puede limitarse al P99.5 para que el cuerpo de la distribución sea legible; las tablas y estadísticas siempre usan el máximo real.

In [ ]:
# Shared plotting helpers and stable palette from Matplotlib's cycle.
colors=plt.rcParams["axes.prop_cycle"].by_key()["color"]
BASE_COLOR,ADAPTED_COLOR=colors[0],colors[1]
def save_plot(fig,filename):
    fig.savefig(OUTPUT_DIR/filename,dpi=180,bbox_inches="tight")
    plt.show()

def ecdf(values):
    x=np.sort(np.asarray(values))
    y=np.arange(1,len(x)+1)/len(x) if len(x) else np.array([])
    return x,y

# 01 — BASE/ADAPTED length histograms.
fig,ax=plt.subplots(figsize=(10,6))
clip=float(np.quantile(np.concatenate([base_full_lengths,adapted_full_lengths]),.995))
bins=np.linspace(0,clip,70)
ax.hist(base_full_lengths,bins=bins,alpha=.5,label=f"BASE (median {np.median(base_full_lengths):.0f})",color=BASE_COLOR)
ax.hist(adapted_full_lengths,bins=bins,alpha=.5,label=f"ADAPTED (median {np.median(adapted_full_lengths):.0f})",color=ADAPTED_COLOR)
for x in [128,256,512,1024]: ax.axvline(x,color="gray",linestyle="--",linewidth=.9,label=str(x))
ax.set(title="Longitud de secuencia ChemBERTa antes y después",xlabel="Tokens incluyendo special tokens (eje limitado a P99.5)",ylabel="Número de secuencias")
ax.legend(ncol=2); ax.grid(alpha=.18); fig.tight_layout(); save_plot(fig,"01_base_vs_adapted_length_histogram.png")

# 02 — ECDF.
fig,ax=plt.subplots(figsize=(10,6))
xb,yb=ecdf(base_full_lengths); xa,ya=ecdf(adapted_full_lengths)
ax.plot(xb,yb,label=f"BASE: {100*np.mean(base_full_lengths<=512):.2f}% ≤512",color=BASE_COLOR)
ax.plot(xa,ya,label=f"ADAPTED: {100*np.mean(adapted_full_lengths<=512):.2f}% ≤512",color=ADAPTED_COLOR)
ax.axvline(512,color="black",linestyle="--",label="contexto 512")
ax.set(title="ECDF de longitud tokenizada",xlabel="Tokens HF con special tokens",ylabel="Fracción de secuencias ≤ x",ylim=(0,1.01))
ax.grid(alpha=.2); ax.legend(); save_plot(fig,"02_base_vs_adapted_length_ecdf.png")

# 03 — fertility.
fig,ax=plt.subplots(figsize=(10,6)); bins_f=np.linspace(0,max(4,float(np.quantile(np.r_[base_fertility_per_sequence,adapted_fertility_per_sequence],.995))),60)
ax.hist(base_fertility_per_sequence,bins=bins_f,alpha=.5,label=f"BASE median={np.median(base_fertility_per_sequence):.2f}",color=BASE_COLOR)
ax.hist(adapted_fertility_per_sequence,bins=bins_f,alpha=.5,label=f"ADAPTED median={np.median(adapted_fertility_per_sequence):.2f}",color=ADAPTED_COLOR)
for x in [1.0,1.2,np.median(base_fertility_per_sequence),np.median(adapted_fertility_per_sequence)]: ax.axvline(x,linestyle="--",linewidth=.9,label=f"{x:.2f}")
ax.set(title="Fertilidad tokenizer por secuencia",xlabel="Subtokens de contenido / lexemas SLICES",ylabel="Secuencias")
ax.grid(alpha=.18); ax.legend(ncol=2); save_plot(fig,"03_base_vs_adapted_fertility.png")

# 04 — overflow grouped bars.
limits=[128,256,512,1024,2048]
base_over=[100*np.mean(base_full_lengths>x) for x in limits]; adapted_over=[100*np.mean(adapted_full_lengths>x) for x in limits]
pos=np.arange(len(limits)); width=.38
fig,ax=plt.subplots(figsize=(10,6)); b1=ax.bar(pos-width/2,base_over,width,label="BASE",color=BASE_COLOR); b2=ax.bar(pos+width/2,adapted_over,width,label="ADAPTED",color=ADAPTED_COLOR)
ax.bar_label(b1,fmt="%.1f%%",padding=2,fontsize=8); ax.bar_label(b2,fmt="%.1f%%",padding=2,fontsize=8)
ax.set_xticks(pos,limits); ax.set(title="Secuencias que exceden cada contexto",xlabel="Threshold de tokens",ylabel="Secuencias > threshold (%)"); ax.grid(axis="y",alpha=.2); ax.legend(); save_plot(fig,"04_context_overflow_comparison.png")

# 05 — token type one-piece alignment.
type_names=["element","integer","periodic_direction","other"]
base_cov=[base_isolated_alignment_by_type[x] for x in type_names]; adapted_cov=[adapted_isolated_alignment_by_type[x] for x in type_names]
pos=np.arange(len(type_names)); fig,ax=plt.subplots(figsize=(10,6)); ax.bar(pos-width/2,base_cov,width,label="BASE",color=BASE_COLOR); ax.bar(pos+width/2,adapted_cov,width,label="ADAPTED",color=ADAPTED_COLOR)
ax.axhline(95,color="black",linestyle="--",label="target 95%")
ax.set_xticks(pos,["element","integer","periodic direction","other"]); ax.set_ylim(0,105); ax.set(title="Ocurrencias alineadas a una pieza en contexto",ylabel="Ocurrencias lexicales (%)"); ax.grid(axis="y",alpha=.2); ax.legend(); save_plot(fig,"05_fragmentation_by_token_type.png")

# 06 — nsites vs token length, common reproducible sample.
scatter_ids=df.sample(n=min(PLOT_SAMPLE_SIZE,len(df)),random_state=RANDOM_SEED).index
fig,ax=plt.subplots(figsize=(9,6)); ax.scatter(df.loc[scatter_ids,"nsites"],df.loc[scatter_ids,"base_hf_tokens_with_special"],s=8,alpha=.22,label="BASE",color=BASE_COLOR); ax.scatter(df.loc[scatter_ids,"nsites"],df.loc[scatter_ids,"adapted_hf_tokens_with_special"],s=8,alpha=.22,label="ADAPTED",color=ADAPTED_COLOR)
ax.axhline(512,color="black",linestyle="--",label="512"); ax.set(title="Tamaño de estructura frente a longitud tokenizada",xlabel="nsites",ylabel="HF tokens, specials incluidos"); ax.grid(alpha=.18); ax.legend(); save_plot(fig,"06_nsites_vs_token_length.png")

# 07 — median lengths by size.
fig,ax=plt.subplots(figsize=(10,6)); pos=np.arange(len(base_vs_adapted_by_nsites)); ax.bar(pos-width/2,base_vs_adapted_by_nsites["base_median_tokens"],width,label="BASE",color=BASE_COLOR); ax.bar(pos+width/2,base_vs_adapted_by_nsites["adapted_median_tokens"],width,label="ADAPTED",color=ADAPTED_COLOR); ax.axhline(512,color="black",linestyle="--",label="512")
ax.set_xticks(pos,base_vs_adapted_by_nsites["nsites_group"]); ax.set(title="Mediana de longitud por grupo nsites",xlabel="nsites group",ylabel="Mediana HF tokens"); ax.grid(axis="y",alpha=.2); ax.legend(); save_plot(fig,"07_median_length_by_nsites_group.png")

# 08 — overflow by size.
fig,ax=plt.subplots(figsize=(10,6)); ax.bar(pos-width/2,base_vs_adapted_by_nsites["base_percent_over_512"],width,label="BASE",color=BASE_COLOR); ax.bar(pos+width/2,base_vs_adapted_by_nsites["adapted_percent_over_512"],width,label="ADAPTED",color=ADAPTED_COLOR); ax.set_xticks(pos,base_vs_adapted_by_nsites["nsites_group"]); ax.set_ylim(0,105); ax.set(title="Overflow >512 por grupo nsites",xlabel="nsites group",ylabel="Secuencias >512 (%)"); ax.grid(axis="y",alpha=.2); ax.legend(); save_plot(fig,"08_over512_by_nsites_group.png")

# 09 — frequency vs saved subtokens.
fig,ax=plt.subplots(figsize=(10,6))
for kind,group in top_tokenization_improvements.groupby("token_type"):
    positive=group.loc[group["estimated_total_subtokens_saved"]>0]
    if not positive.empty: ax.scatter(np.log10(positive["frequency"]),positive["subtokens_saved_per_occurrence"],s=18,alpha=.65,label=kind)
for token in ["ooo","+oo","-oo","o-o","o+o"]:
    hit=top_tokenization_improvements.loc[top_tokenization_improvements["token"]==token]
    if not hit.empty and hit.iloc[0]["estimated_total_subtokens_saved"]>0:
        row=hit.iloc[0]; ax.annotate(token,(np.log10(row["frequency"]),row["subtokens_saved_per_occurrence"]),fontsize=8)
ax.set(title="Frecuencia lexical y ahorro estimado",xlabel="log10(frecuencia SLICES)",ylabel="Subtokens ahorrados por ocurrencia"); ax.grid(alpha=.2); ax.legend(); save_plot(fig,"09_token_frequency_vs_subtoken_savings.png")

# 10 — cumulative savings.
savings=top_tokenization_improvements.loc[top_tokenization_improvements["estimated_total_subtokens_saved"]>0].sort_values("estimated_total_subtokens_saved",ascending=False)
fig,ax=plt.subplots(figsize=(10,6))
if not savings.empty:
    cumulative=savings["estimated_total_subtokens_saved"].cumsum()/savings["estimated_total_subtokens_saved"].sum()*100
    ax.plot(np.arange(1,len(savings)+1),cumulative,color=BASE_COLOR)
ax.set(title="Ahorro acumulado atribuido a lexemas",xlabel="Número de lexemas ordenados por ahorro",ylabel="Ahorro acumulado (%)",ylim=(0,105)); ax.grid(alpha=.2); save_plot(fig,"10_cumulative_token_savings.png")

# 11 — natural vs BASE vs ADAPTED distribution, fliers hidden for legibility.
fig,ax=plt.subplots(figsize=(10,6)); box=ax.boxplot([df["slices_whitespace_tokens"],base_full_lengths,adapted_full_lengths],labels=["SLICES lexical","BASE HF","ADAPTED HF"],showfliers=False,patch_artist=True)
for patch,color in zip(box["boxes"],[colors[2],BASE_COLOR,ADAPTED_COLOR]): patch.set_facecolor(color); patch.set_alpha(.6)
ax.set(title="Longitud lexical natural frente a tokenizers (outliers ocultos)",ylabel="Tokens por secuencia"); ax.grid(axis="y",alpha=.2)
summary_lines=[f"{name}: median={np.median(arr):.0f}, P95={np.quantile(arr,.95):.0f}, P99={np.quantile(arr,.99):.0f}" for name,arr in [("Natural",df["slices_whitespace_tokens"]),("BASE",base_full_lengths),("ADAPTED",adapted_full_lengths)]]
ax.text(.02,.98,"\n".join(summary_lines),transform=ax.transAxes,va="top",fontsize=9,bbox={"facecolor":"white","alpha":.8})
save_plot(fig,"11_natural_vs_base_vs_adapted_lengths.png")

# 12 — compact tokenization examples: short, near context, high fertility, representative token types.
example_indices=[]
example_indices.append(int(df["slices_whitespace_tokens"].idxmin()))
example_indices.append(int((df["base_hf_tokens_with_special"]-512).abs().idxmin()))
example_indices.append(int(df["base_fertility"].idxmax()))
example_indices.append(int(df["base_hf_tokens_with_special"].idxmax()))
example_indices=list(dict.fromkeys(example_indices))[:4]
example_tokens=[]
for kind in ["element","periodic_direction","integer"]:
    subset=top_tokenization_improvements.loc[top_tokenization_improvements["token_type"]==kind]
    if not subset.empty:
        example_tokens.append(subset.sort_values("frequency",ascending=False).iloc[0]["token"])
example_token_blocks=[]
for token in example_tokens:
    row=base_vs_adapted_fragmentation.loc[base_vs_adapted_fragmentation["token"]==token].iloc[0]
    example_token_blocks.append((token,row["token_type"],decode_json_tokens(row["base_context_tokens"]),decode_json_tokens(row["adapted_context_tokens"])))
fig,ax=plt.subplots(figsize=(14,9)); ax.axis("off")
lines=["TOKENIZATION EXAMPLES (textual IDs shown as token pieces)",""]
for label,index in enumerate(example_indices,1):
    row=df.loc[index]
    snippet=" ".join(row["slices_normalized"].split()[:18])
    b=base_tokenizer.convert_ids_to_tokens(base_tokenizer.encode(snippet,add_special_tokens=False))
    a=adapted_tokenizer.convert_ids_to_tokens(adapted_tokenizer.encode(snippet,add_special_tokens=False))
    lines += [f"Sequence {label}: nsites={row['nsites']}, SLICES lexical={row['slices_whitespace_tokens']}, BASE={row['base_hf_tokens_with_special']}, ADAPTED={row['adapted_hf_tokens_with_special']}",
              f"SLICES prefix: {snippet}",f"BASE pieces: {' | '.join(b[:45])}",f"ADAPTED pieces: {' | '.join(a[:45])}",""]
for token,kind,b,a in example_token_blocks:
    lines += [f"Lexeme {token!r} ({kind}) in whitespace context:",f"BASE: {b}",f"ADAPTED: {a}",""]
ax.text(.01,.99,"\n".join(lines),va="top",ha="left",family="monospace",fontsize=8,wrap=True)
fig.tight_layout(); save_plot(fig,"12_tokenization_examples.png")

# 13 — dashboard summary.
fig,ax=plt.subplots(figsize=(12,7)); ax.axis("off")
dashboard_rows=[
 ["Metric","BASE","ADAPTED"],
 ["Global fertility",f"{base_final_metrics['global_fertility']:.3f}",f"{adapted_metrics['global_fertility']:.3f}"],
 ["Median HF length",f"{base_final_metrics['median_hf_tokens']:.0f}",f"{adapted_metrics['median_hf_tokens']:.0f}"],
 ["P95 HF length",f"{base_final_metrics['p95_hf_tokens']:.0f}",f"{adapted_metrics['p95_hf_tokens']:.0f}"],
 ["% >512",f"{base_final_metrics['percent_over_512']:.2f}%",f"{adapted_metrics['percent_over_512']:.2f}%"],
 ["Element 1-piece",f"{base_isolated_alignment_by_type['element']:.2f}%",f"{adapted_isolated_alignment_by_type['element']:.2f}%"],
 ["Periodic 1-piece",f"{base_isolated_alignment_by_type['periodic_direction']:.2f}%",f"{adapted_isolated_alignment_by_type['periodic_direction']:.2f}%"],
 ["Integer 1-piece",f"{base_isolated_alignment_by_type['integer']:.2f}%",f"{adapted_isolated_alignment_by_type['integer']:.2f}%"],
 ["UNK sequence rate",f"{base_unk_rate:.2f}%" if base_unk_rate is not None else "N/A",f"{adapted_unk_rate:.2f}%" if adapted_unk_rate is not None else "N/A"],
]
table=ax.table(cellText=dashboard_rows[1:],colLabels=dashboard_rows[0],loc="center",cellLoc="center",colLoc="center")
table.auto_set_font_size(False); table.set_fontsize(13); table.scale(1,2)
ax.set_title("ChemBERTa–SLICES tokenizer adaptation summary",fontsize=16,pad=25)
ax.text(.5,.08,f"Added lexical IDs: {n_added_tokens} | Strategy: {adaptation_strategy}",ha="center",transform=ax.transAxes,fontsize=12)
fig.tight_layout(); save_plot(fig,"13_tokenizer_adaptation_summary.png")

# 14 — simulated retained fraction at context 512.
base_retained=np.minimum(base_full_lengths,512)/base_full_lengths
adapted_retained=np.minimum(adapted_full_lengths,512)/adapted_full_lengths
fig,ax=plt.subplots(figsize=(10,6)); bins=np.linspace(0,1,41)
ax.hist(base_retained,bins=bins,alpha=.5,label=f"BASE median={np.median(base_retained):.3f}",color=BASE_COLOR)
ax.hist(adapted_retained,bins=bins,alpha=.5,label=f"ADAPTED median={np.median(adapted_retained):.3f}",color=ADAPTED_COLOR)
ax.set(title="Fracción de longitud retenida si el contexto fuera 512 (simulación)",xlabel="min(length,512) / length",ylabel="Secuencias"); ax.grid(alpha=.18); ax.legend(); save_plot(fig,"14_truncation_retained_fraction.png")


## 25. Criterios CRITICAL, TARGET y DESIRABLE

Los checks **CRITICAL** deciden si el tokenizer se puede considerar íntegro: IDs originales, special IDs, UNK no peor, reproducibilidad save/reload y mismo número de secuencias. Un fallo en cualquiera rechaza la adaptación. Los objetivos TARGET son metas experimentales del proyecto (no estándares universales); el score es la fracción de esos criterios cumplidos. DESIRABLE registra alineación lexical ≥95 % y longitud cercana al número natural de lexemas, pero no se mezcla en el target score.

In [ ]:
n_adapted_sequences=len(adapted_full_lengths)
critical_checks={
    "C1_original_vocab_ids_preserved":original_vocab_preserved,
    "C2_special_ids_preserved":special_ids_preserved,
    "C3_unk_rate_no_worse":(adapted_unk_rate is not None and base_unk_rate is not None and adapted_unk_rate<=base_unk_rate),
    "C4_save_reload_100_percent":reload_reproducible,
    "C5_no_sequence_loss":n_adapted_sequences==len(base_full_lengths),
}
relative_fertility_reduction=(base_final_metrics["global_fertility"]-adapted_metrics["global_fertility"])/base_final_metrics["global_fertility"] if base_final_metrics["global_fertility"] else np.nan
median_length_relative_reduction=(base_final_metrics["median_hf_tokens"]-adapted_metrics["median_hf_tokens"])/base_final_metrics["median_hf_tokens"] if base_final_metrics["median_hf_tokens"] else np.nan

target_checks={
    "T1_global_fertility_le_1_20":adapted_metrics["global_fertility"]<=1.20,
    "T2_median_fertility_le_1_15":adapted_metrics["median_fertility"]<=1.15,
    "T3_P95_fertility_le_1_25":adapted_metrics["p95_fertility"]<=1.25,
    "T4_periodic_alignment_ge_95_percent":adapted_alignment_by_type["periodic_direction"]>=95,
    "T5_element_alignment_ge_95_percent":adapted_alignment_by_type["element"]>=95,
    "T6_integer_alignment_ge_95_percent":adapted_alignment_by_type["integer"]>=95,
    "T7_relative_over512_reduction_ge_70_percent":over512_relative_reduction>=.70,
}
desirable_checks={
    "D1_global_lexical_alignment_ge_95_percent":adapted_alignment_by_type["global"]>=95,
    "D2_median_length_approaches_110_to_130":110<=adapted_metrics["median_hf_tokens"]<=130,
    "D3_P95_length_below_450":adapted_metrics["p95_hf_tokens"]<450,
}
critical_pass=all(critical_checks.values())
targets_passed=sum(target_checks.values())
total_targets=len(target_checks)
target_score=targets_passed/total_targets if total_targets else 0.0
if not critical_pass:
    final_recommendation="DO NOT USE — TOKENIZER ADAPTATION NOT ACCEPTED (critical check failed)"
elif target_score>=.80:
    final_recommendation="ACCEPT FOR DAPT"
elif target_score>=.60:
    final_recommendation="ACCEPT WITH CAUTION / REVIEW"
else:
    final_recommendation="DO NOT USE — EVALUATE DEDICATED SLICES TOKENIZER"

criterion_rows=[]
criterion_values={
    "C1_original_vocab_ids_preserved":(n_original_ids_changed,0,"0 changed original token IDs"),
    "C2_special_ids_preserved":(len(changed_special_ids),0,"0 changed special token IDs"),
    "C3_unk_rate_no_worse":(base_unk_rate,adapted_unk_rate,"adapted UNK rate <= baseline"),
    "C4_save_reload_100_percent":(None,100.0 if reload_reproducible else 0.0,"100% exact sequence encoding agreement"),
    "C5_no_sequence_loss":(len(base_full_lengths),n_adapted_sequences,"same number of sequences"),
    "T1_global_fertility_le_1_20":(base_final_metrics["global_fertility"],adapted_metrics["global_fertility"],"<= 1.20"),
    "T2_median_fertility_le_1_15":(base_final_metrics["median_fertility"],adapted_metrics["median_fertility"],"<= 1.15"),
    "T3_P95_fertility_le_1_25":(base_final_metrics["p95_fertility"],adapted_metrics["p95_fertility"],"<= 1.25"),
    "T4_periodic_alignment_ge_95_percent":(base_alignment_by_type["periodic_direction"],adapted_alignment_by_type["periodic_direction"],">= 95%"),
    "T5_element_alignment_ge_95_percent":(base_alignment_by_type["element"],adapted_alignment_by_type["element"],">= 95%"),
    "T6_integer_alignment_ge_95_percent":(base_alignment_by_type["integer"],adapted_alignment_by_type["integer"],">= 95%"),
    "T7_relative_over512_reduction_ge_70_percent":(base_final_metrics["percent_over_512"],adapted_metrics["percent_over_512"],">= 70% relative reduction"),
    "D1_global_lexical_alignment_ge_95_percent":(base_alignment_by_type["global"],adapted_alignment_by_type["global"],">= 95%"),
    "D2_median_length_approaches_110_to_130":(base_final_metrics["median_hf_tokens"],adapted_metrics["median_hf_tokens"],"110-130 tokens"),
    "D3_P95_length_below_450":(base_final_metrics["p95_hf_tokens"],adapted_metrics["p95_hf_tokens"],"< 450 tokens"),
}
for category,checks in [("CRITICAL",critical_checks),("TARGET",target_checks),("DESIRABLE",desirable_checks)]:
    for name,passed in checks.items():
        baseline_value,adapted_value,target_value=criterion_values[name]
        criterion_rows.append({"criterion":name,"category":category,"baseline":baseline_value,"adapted":adapted_value,"target":target_value,"passed":bool(passed),"comment":"Targets are project-specific review criteria, not universal standards." if category!="CRITICAL" else "Tokenizer integrity invariant."})


## 26. Tamaño final y reporte de aceptación

Se separa el crecimiento real del vocabulario del número de lexemas que recibieron metadatos `AddedToken`: un lexema existente puede conservar su ID y no aumentar `len(tokenizer)`. Los IDs nuevos deben quedar después de los IDs originales. La decisión final combina integridad obligatoria y porcentaje de objetivos experimentales cumplidos.

In [ ]:
percentage_vocab_growth=100*n_added_tokens/base_len if base_len else 0.0
size_summary=pd.DataFrame([{
    "base_vocab_size":base_vocab_size,
    "base_len_tokenizer":base_len,
    "n_added_tokens":n_added_tokens,
    "adapted_vocab_size":adapted_len,
    "percentage_vocab_growth":percentage_vocab_growth,
    "new_token_id_min":new_token_id_min,
    "new_token_id_max":new_token_id_max,
    "new_ids_after_base":new_ids_after_base,
}])
display(size_summary)
tokenizer_adaptation_report=pd.DataFrame(criterion_rows)
tokenizer_adaptation_report["target_score"] = target_score
tokenizer_adaptation_report["critical_pass"] = critical_pass
display(tokenizer_adaptation_report)
print("Critical checks:",critical_checks)
print("Target criteria passed:",targets_passed,"/",total_targets,"=",f"{target_score:.1%}")
print("Desirable checks:",desirable_checks)
print("Recommendation:",final_recommendation)


## 27. Guardar resultados y tokenizer aprobado

Los CSV/PNG/JSON siempre se guardan en la carpeta de esta ejecución. El tokenizer candidato ya se guardó ahí para auditoría. Solo si todos los checks CRITICAL pasan **y** se obtiene al menos 60 % del target score (aceptación o aceptación con cautela) se copia a `audit/tokenizers/chemberta_zinc_slices_adapted/`. Si la aceptación se confirma, el tokenizer oficial se guarda en la ruta canónica. Una nueva corrida aceptada actualiza allí los archivos del tokenizer; una corrida rechazada deja intacta la última versión aceptada. Si falla algún check crítico o el score queda por debajo de 60 %, no se publica como tokenizer oficial. No se guarda ningún peso de modelo.

In [ ]:
# Tablas del run; nunca se sobrescriben los outputs del audit 01.
OUTPUT_TABLES={
    "base_vs_adapted_metrics.csv":base_vs_adapted_metrics,
    "base_vs_adapted_by_nsites.csv":base_vs_adapted_by_nsites,
    "base_vs_adapted_token_types.csv":base_vs_adapted_token_types,
    "base_vs_adapted_truncation_512.csv":base_vs_adapted_truncation_512,
    "tokens_selected_for_adaptation.csv":tokens_selected_for_adaptation,
    "top_tokenization_improvements.csv":top_tokenization_improvements,
    "remaining_fragmented_tokens.csv":remaining_fragmented_tokens,
    "longest_sequences_comparison.csv":longest_sequences_comparison,
    "tokenizer_adaptation_report.csv":tokenizer_adaptation_report,
    "slices_lexical_vocabulary.csv":lexical_vocab,
    "base_lexical_fragmentation.csv":base_fragmentation,
    "base_vs_adapted_lexical_fragmentation.csv":base_vs_adapted_fragmentation,
    "strategy_comparison.csv":strategy_comparison,
    "base_whitespace_marker_comparison.csv":space_marker_comparison,
    "baseline_reconciliation.csv":baseline_reconciliation,
    "new_token_ids.csv":pd.DataFrame([{"token":token,"token_id":int(adapted_vocab[token])} for token in new_lexical_tokens]),
}
for filename,table in OUTPUT_TABLES.items(): table.to_csv(OUTPUT_DIR/filename,index=False)

# La vista de secuencias más largas omite cadenas completas para mantener el archivo compacto;
# el source_row_index y mat_id permiten recuperarlas del CSV fuente.
longest_sequences_comparison.to_csv(OUTPUT_DIR/"longest_sequences_comparison.csv",index=False)

# Registrar cada decisión y checks con valores observados.
base_percentages={k:v for k,v in base_final_metrics.items()}
adapted_percentages={k:v for k,v in adapted_metrics.items()}

def to_jsonable(value):
    if value is None: return None
    if isinstance(value,(np.integer,)): return int(value)
    if isinstance(value,(np.floating,)): return float(value) if np.isfinite(value) else None
    if isinstance(value,(np.bool_,)): return bool(value)
    return value

adaptation_summary={
    "base_tokenizer":BASE_MODEL_NAME,
    "model_revision_requested":MODEL_REVISION,
    "resolved_tokenizer_revision":getattr(base_tokenizer,"init_kwargs",{}).get("_commit_hash"),
    "n_sequences":int(len(df)),
    "n_unique_slices_tokens":int(n_unique_slices_tokens),
    "base_vocab_size":int(base_len),
    "adapted_vocab_size":int(adapted_len),
    "n_added_tokens":int(n_added_tokens),
    "n_candidate_lexemes":int(len(candidate_tokens)),
    "adaptation_strategy":adaptation_strategy,
    "strategy_A_lstrip_false_metrics":strategy_comparison.loc[strategy_comparison["strategy"]=="A_lstrip_false"].iloc[0].to_dict(),
    "strategy_B_lstrip_true_metrics":strategy_comparison.loc[strategy_comparison["strategy"]=="B_lstrip_true"].iloc[0].to_dict(),
    "base":base_final_metrics,
    "adapted":adapted_metrics,
    "base_lexical_alignment_percent":base_alignment_by_type,
    "adapted_lexical_alignment_percent":adapted_alignment_by_type,
    "base_isolated_single_token_percent":base_isolated_alignment_by_type,
    "adapted_isolated_single_token_percent":adapted_isolated_alignment_by_type,
    "improvement":{
        "fertility_relative_reduction":relative_fertility_reduction,
        "median_length_relative_reduction":median_length_relative_reduction,
        "over512_absolute_percentage_point_reduction":over512_reduction_pp,
        "over512_relative_reduction":over512_relative_reduction,
        "space_marker_reduction_percent":space_marker_reduction_percent,
    },
    "vocabulary_size":{
        "base_len":int(base_len),"adapted_len":int(adapted_len),
        "n_added_ids":int(n_added_tokens),"percentage_growth":percentage_vocab_growth,
        "new_token_id_min":new_token_id_min,"new_token_id_max":new_token_id_max,
    },
    "validation":{
        "original_ids_preserved":original_vocab_preserved,
        "n_original_ids_changed":int(n_original_ids_changed),
        "special_ids_preserved":special_ids_preserved,
        "n_special_ids_changed":int(len(changed_special_ids)),
        "reload_reproducible":reload_reproducible,
        "reload_test_sequences":int(reload_n),
        "reload_equal_encodings":int(sum(reload_equal)),
        "critical_pass":critical_pass,
        "target_score":target_score,
        "targets_passed":int(targets_passed),"total_targets":int(total_targets),
    },
    "accepted_for_dapt":bool(critical_pass and target_score>=.60),
    "final_recommendation":final_recommendation,
    "python_version":sys.version,
    "transformers_version":transformers.__version__,
    "pandas_version":pd.__version__,
    "numpy_version":np.__version__,
    "batch_size":BATCH_SIZE,
    "strategy_sample_size":int(len(strategy_sample)),
    "random_seed":RANDOM_SEED,
    "truncation":False,"padding":False,"transformer_weights_loaded":False,
    "energy_targets_loaded":False,
    "output_directory":str(OUTPUT_DIR),
}
with (OUTPUT_DIR/"tokenizer_adaptation_summary.json").open("w",encoding="utf-8") as handle:
    json.dump(adaptation_summary,handle,ensure_ascii=False,indent=2,default=to_jsonable)

accepted_for_dapt=bool(critical_pass and target_score>=.60)
official_tokenizer_path=None
if accepted_for_dapt:
    destination=TOKENIZER_DIR
    destination.mkdir(parents=True,exist_ok=True)
    adapted_tokenizer.save_pretrained(destination)
    manifest={
        "base_model":BASE_MODEL_NAME,"base_vocab_size":int(base_len),
        "adapted_vocab_size":int(adapted_len),"n_added_tokens":int(n_added_tokens),
        "strategy":adaptation_strategy,"lstrip":selected_lstrip,
        "dataset_sequences":int(len(df)),"unique_slices_tokens":int(n_unique_slices_tokens),
        "base_global_fertility":base_final_metrics["global_fertility"],
        "adapted_global_fertility":adapted_metrics["global_fertility"],
        "base_percent_over_512":base_final_metrics["percent_over_512"],
        "adapted_percent_over_512":adapted_metrics["percent_over_512"],
        "original_token_ids_preserved":original_vocab_preserved,
        "special_token_ids_preserved":special_ids_preserved,
        "reload_reproducible":reload_reproducible,
        "accepted_for_dapt":accepted_for_dapt,
        "final_recommendation":final_recommendation,
        "creation_timestamp":datetime.now().isoformat(timespec="seconds"),
        "transformers_version":transformers.__version__,
        "run_summary":str(OUTPUT_DIR/"tokenizer_adaptation_summary.json"),
    }
    with (destination/"adaptation_manifest.json").open("w",encoding="utf-8") as handle:
        json.dump(manifest,handle,ensure_ascii=False,indent=2,default=to_jsonable)
    official_tokenizer_path=destination
    print("Tokenizer adaptado guardado en:",official_tokenizer_path)
else:
    print("No se guardó tokenizer oficial; el candidato diagnóstico queda en:",CANDIDATE_TOKENIZER_DIR)


## 28. Resumen final y decisión

La recomendación separa: integridad (CRITICAL), número de objetivos del proyecto cumplidos (TARGET) y preferencias de calidad (DESIRABLE). Un resultado favorable solo indica que la extensión conserva compatibilidad del vocabulario y mejora métricas de eficiencia/contexto según estos criterios. No demuestra una menor MLM loss ni mejoras de predicción energética, química o ranking de polimorfos.

In [ ]:
print("================================================================")
print("CHEMBERTA-SLICES TOKENIZER ADAPTATION")
print("================================================================")
print("\nCorpus")
print(f"  SLICES sequences:                        {len(df):>10}")
print(f"  Unique SLICES lexical tokens:            {n_unique_slices_tokens:>10}")
print("\nVocabulary")
print(f"  BASE effective vocabulary:              {base_len:>10}")
print(f"  Added SLICES IDs:                        {n_added_tokens:>10}")
print(f"  ADAPTED effective vocabulary:            {adapted_len:>10}")
print(f"  Vocabulary growth:                       {percentage_vocab_growth:.2f}%")
print(f"  New token ID min/max:                    {new_token_id_min} / {new_token_id_max}")
print("\n                              BASE          ADAPTED")
print("----------------------------------------------------------------")
for key,label in [("median_hf_tokens","Median HF tokens"),("p95_hf_tokens","P95 HF tokens"),("p99_hf_tokens","P99 HF tokens"),
                  ("global_fertility","Global fertility"),("median_fertility","Median fertility"),("p95_fertility","P95 fertility"),
                  ("percent_over_512","Sequences >512 (%)"),("percent_over_1024","Sequences >1024 (%)"),("unk_rate","UNK rate (%)")]:
    b=base_final_metrics[key]; a=adapted_metrics[key]
    print(f"{label:<30}{b:>12.3f}{a:>16.3f}")
for kind,label in [("element","Element 1-token (%)"),("periodic_direction","Periodic 1-token (%)"),("integer","Integer 1-token (%)")]:
    print(f"{label:<30}{base_isolated_alignment_by_type[kind]:>12.2f}{adapted_isolated_alignment_by_type[kind]:>16.2f}")
print("\nIntegrity")
print("  Original token IDs preserved:           ","YES" if original_vocab_preserved else "NO")
print("  Special IDs preserved:                  ","YES" if special_ids_preserved else "NO")
print("  Save/reload reproducible:                ","YES" if reload_reproducible else "NO")
print("\nAcceptance criteria")
print("  Critical checks passed:                  ","YES" if critical_pass else "NO")
print(f"  Target criteria passed:                   {targets_passed} / {total_targets}")
print(f"  Target score:                             {target_score:.2%}")
print("\nFINAL RECOMMENDATION:")
print(" ",final_recommendation)
print("\nRun outputs:",OUTPUT_DIR)
if official_tokenizer_path is not None: print("Approved tokenizer:",official_tokenizer_path)
print("================================================================")


## Qué queda para la etapa siguiente

Este notebook termina con la decisión cuantitativa del tokenizer y los artefactos comparativos. No empieza la etapa siguiente. Si el tokenizer se acepta, un notebook futuro —por ejemplo `03_slices_dapt_mlm.ipynb`— podrá cargar el backbone ChemBERTa, redimensionar embeddings e inicializar vectores para nuevos IDs antes de MLM/DAPT. Esa etapa tendrá que medir MLM loss y downstream tasks; una reducción del conteo de tokens por sí sola no garantiza mejor desempeño.
